# 05 · The fibroblast programmes in Xenium: scoring across the morphological compartments

**Question.** Where do the 16 programmes sit in the tissue, and where do the four programmes
followed into chromatin (S1, S7, S12, S15) sit in particular?

**Units.** Per-cell programme scores are not reliable on a 5,000-gene panel (a split-half
reliability test failed in every section in an earlier round), so the unit is the **donor ×
morphological-niche pseudobulk**, built by `Spatial_analyses/.../final/02_pseudobulks.ipynb`
from seven Xenium sections (U51 2 w, U48 6 m, Z16 8 y, Z17 11 y, A67 20 y, A72 33 y, A70 37 y)
and the eight H&E morphology niches (`morpho_niche_l2`: cortex_septal, cortex, medulla,
medulla_hc, adipose, fibrosis_adipose, fibrosis, vessel). Three representations are scored and
read together:

1. **H&E patches** — all transcripts in 56 × 56 µm tiles, segmentation-free; the tissue map;
2. **segmented cells** — all cells of a niche; composition-weighted like the patches;
3. **fibroblast-compartment cells only** — the programme intensity among the fibroblasts that
   are there. Fibroblasts are ~2 % of cells, so a niche can score high in (1) and (2) simply
   because it holds more fibroblasts; only (3) says whether *fibroblasts* differ between niches.

**Signatures.** For every factor, its 30 most *distinctive* genes on the panel (distinctiveness
= normalised loading × the share of the gene's total loading held by the factor), weighted by the
normalised loadings and scored with the decoupler univariate linear model (ULM) on the
log-normalised pseudobulk; `score_genes` is run as a check. Only 31 % of the 3,671-gene basis is
on the panel, so two gates precede any interpretation: the panel-restricted signatures must
reproduce which fibroblast state runs which programme in scRNA-seq (gate 1), and a programme
whose best-scoring Xenium cell type is not a fibroblast state is carried by another lineage on
this panel and its niche map is not fibroblast biology (gate 2).

**Statistics.** Scores are centred within donor across niches before averaging (sections differ
several-fold in depth; the question is which niche is high *within* a section). Dot area on every
dotplot is the fraction of donors agreeing with the sign of the mean — with 2–3 sections per age
group that is the number that decides whether a niche effect is real. Donor agreement on the
niche *ordering* is Kendall's W with a permutation p (niche labels permuted within donors,
Benjamini–Hochberg within age group). Age is adult (3 sections) versus young (4); the exact
two-sided p cannot fall below 2/35, so age effects are reported as differences of means and
rank-biserial effect sizes.

In [1]:
import os, sys, glob, itertools, warnings
warnings.filterwarnings('ignore')
import hdf5plugin
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import decoupler as dc
from scipy.stats import kruskal, mannwhitneyu, spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '/nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/notebooks/revisions/final')
import fb_common as C

C.set_style()
sc.settings.verbosity = 0
RES, PLT = C.out_dirs('05_xenium_spatial')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

FACTORS, LAB, NAMES = C.FACTORS, C.FACTOR_LABELS, C.FACTOR_NAMES
SELECTED = [C.PROGRAMMES[p] for p in C.PROGRAMMES]                        # S1, S7, S12, S15
N_GENES_SIG, MIN_GENES, TMIN = 30, 8, 5
ANNOTATION = f'{C.RESULTS}/02_inmf_model/factor_annotation_{C.TAG}.csv'
print(f'decoupler {dc.__version__} | scanpy {sc.__version__}')

decoupler 2.2.0 | scanpy 1.11.5


## 1 · The pseudobulks (from the Xenium final pipeline, notebook 02)

In [2]:
UNITS = [('patches', 'morpho_niche_l2'), ('patches', 'morpho_niche_l3'),
         ('cells', 'morpho_niche_l2'), ('cells', 'morpho_niche_l3'), ('cells', 'taa_l3_upd'),
         ('fbcells', 'morpho_niche_l2'), ('fbcells', 'morpho_niche_l3')]
NICHE_UNITS = [u for u in UNITS if u[1].startswith('morpho')]
META = ['donor', 'group', 'level', 'representation', 'age_group', 'age_group2', 'age_years', 'psbulk_cells', 'psbulk_counts_shared']


def load_pb(rep, level):
    a = ad.read_h5ad(f'{C.PB_DIR}/pb_{rep}_{level}.h5ad')
    a = a[:, a.var['shared_panel'].to_numpy()].copy()        # the 4,993 genes on the v1 panel and in the cell object
    a.obs['age_group'] = pd.Categorical(a.obs['age_group'].astype(str), categories=C.AGE_ORDER)
    a.obs['age_group2'] = pd.Categorical(a.obs['age_group2'].astype(str), categories=['young', 'adult'])
    return a


PBS = {u: load_pb(*u) for u in UNITS}
PANEL = PBS[('cells', 'morpho_niche_l2')].var_names
assert all(list(a.var_names) == list(PANEL) for a in PBS.values())
for (rep, lv), a in PBS.items():
    print(f'{rep:8s} {lv:16s} {a.n_obs:4d} samples | {a.obs["group"].nunique():3d} groups | {a.obs["donor"].nunique()} donors')
print(f'\nscoring universe: {len(PANEL):,} genes')


def long_table(scores, obs, sources, value='score'):
    m = scores[sources].stack().rename(value).rename_axis(['sample', 'source']).reset_index()
    return m.join(obs[META], on='sample')

patches  morpho_niche_l2    56 samples |   8 groups | 7 donors
patches  morpho_niche_l3    82 samples |  13 groups | 7 donors
cells    morpho_niche_l2    56 samples |   8 groups | 7 donors
cells    morpho_niche_l3    96 samples |  14 groups | 7 donors
cells    taa_l3_upd        316 samples |  51 groups | 7 donors
fbcells  morpho_niche_l2    46 samples |   8 groups | 7 donors
fbcells  morpho_niche_l3    68 samples |  13 groups | 7 donors

scoring universe: 4,993 genes


## 2 · Signatures: the most distinctive panel genes of every factor

In [3]:
W = pd.read_csv(f'{C.INMF_DIR}/W_{C.TAG}.csv', index_col=0)
assert list(W.index) == FACTORS and (W.to_numpy() >= 0).all()
W_norm, _ = C.gene_zscores(W)
DIST = C.distinctiveness(W_norm)
on = W.columns.intersection(PANEL)
print(f'factor basis {W.shape[1]:,} genes | panel {len(PANEL):,} | on panel {len(on):,} ({len(on) / W.shape[1]:.1%} of the basis)')

rows, SIGS, WEIGHTS = [], {}, {}
for f in FACTORS:
    row = W_norm.loc[f]
    ranked = DIST.loc[f, on]
    ranked = ranked[ranked > 0].sort_values(ascending=False)
    genes = list(ranked.index[:N_GENES_SIG])
    w = W_norm.loc[f, genes]
    SIGS[f], WEIGHTS[f] = genes, w / w.max()
    rows.append({'factor': f, 'name': NAMES[f], 'n_on_panel': int((row[on] > 0).sum()), 'mass_on_panel': round(float(row[on].sum()), 3),
                 'top50_on_panel': sum(g in set(on) for g in row.nlargest(50).index), 'n_genes': len(genes), 'genes': ', '.join(genes)})
sigtab = pd.DataFrame(rows).set_index('factor')
sigtab.to_csv(f'{RES}/signatures.csv')
print(sigtab.drop(columns='genes').to_string())
assert (sigtab['n_genes'] >= MIN_GENES).all()
OV = pd.DataFrame({a: {b: len(set(SIGS[a]) & set(SIGS[b])) / len(set(SIGS[a]) | set(SIGS[b])) for b in FACTORS} for a in FACTORS})
pairs = sorted([(a, b, OV.loc[a, b]) for a, b in itertools.combinations(FACTORS, 2)], key=lambda x: -x[2])[:5]
print('\nlargest signature overlaps (Jaccard):', ', '.join(f'{a}-{b} {j:.2f}' for a, b, j in pairs))

factor basis 3,671 genes | panel 4,993 | on panel 1,161 (31.6% of the basis)
                               name  n_on_panel  mass_on_panel  top50_on_panel  n_genes
factor                                                                                 
S1         TGFb / hypoxia / mechano         624          0.410              23       30
S2         lymphoid (contamination)         676          0.396              25       30
S3              interlobular (CCN5)         600          0.380              21       30
S4                    proliferation         562          0.513              31       30
S5                      perilobular         558          0.309               8       30
S6               IEG / dissociation         466          0.356              14       30
S7                     inflammatory         539          0.416              22       30
S8            perilobular (nuclear)         665          0.299              15       30
S9              interlobular (PI16)        

## 3 · Scoring every pseudobulk (ULM, with `score_genes` as a check)

In [4]:
NET = pd.DataFrame([{'source': f, 'target': g, 'weight': float(w)} for f, ser in WEIGHTS.items() for g, w in ser.items()])


def score_both(pb):
    p = pb.copy()
    sc.pp.normalize_total(p, target_sum=1e4)
    sc.pp.log1p(p)
    dc.mt.ulm(data=p, net=NET, tmin=TMIN, verbose=False)
    ulm = p.obsm['score_ulm']
    sg = {}
    for f, genes in SIGS.items():
        sc.tl.score_genes(p, gene_list=genes, score_name='_s', use_raw=False, ctrl_size=max(50, len(genes)), n_bins=25, random_state=0)
        sg[f] = p.obs['_s'].to_numpy()
    return ulm, pd.DataFrame(sg, index=p.obs_names)


frames = []
for (rep, lv), a in PBS.items():
    ulm, sg = score_both(a)
    t = long_table(ulm, a.obs, FACTORS)
    t = t.merge(long_table(sg, a.obs, FACTORS, value='score_genes')[['sample', 'source', 'score_genes']], on=['sample', 'source'])
    frames.append(t)
t1 = pd.concat(frames, ignore_index=True)
t1['age_group'] = pd.Categorical(t1['age_group'].astype(str), categories=C.AGE_ORDER)
t1.to_csv(f'{RES}/table1_scores.csv', index=False)
agree = (t1[t1['level'].str.startswith('morpho')].groupby(['representation', 'source'], observed=True)
         .apply(lambda d: spearmanr(d['score'], d['score_genes']).statistic, include_groups=False).unstack('representation').reindex(FACTORS))
agree.to_csv(f'{RES}/scorer_agreement.csv')
print(f'{len(t1):,} sample x factor rows | ULM vs score_genes Spearman over donor x niche samples: '
      + ', '.join(f'{c} median {agree[c].median():.2f} (min {agree[c].min():.2f})' for c in agree.columns))

11,520 sample x factor rows | ULM vs score_genes Spearman over donor x niche samples: cells median 0.98 (min 0.87), fbcells median 0.97 (min 0.90), patches median 0.96 (min 0.91)


## 4 · Gate 1 — do the panel-restricted signatures recover the fibroblast states?

scRNA-seq side: mean usage share of every factor per harmonised fibroblast state (`taa_l3_harmonized`,
cells arm, from the model's H matrix). Xenium side: the donor × cell-type pseudobulk on the same
state vocabulary (`taa_l3_upd`), ULM centred within donor. Both are centred per factor across the
states, so shape is compared, never level.

In [5]:
def centre_within_donor(t, value='score', by=('representation', 'level', 'source')):
    out = t.copy()
    out[value + '_centred'] = t[value] - t.groupby(list(by) + ['donor'], observed=True)[value].transform('mean')
    return out


light = ad.read_h5ad(C.light_path())
H = pd.read_parquet(f'{C.INMF_DIR}/H_{C.TAG}.parquet')
H = H.loc[H.index.intersection(light.obs_names)]
harm = pd.read_csv(C.HARMONIZED_OBS, index_col=0, usecols=lambda c: c in ('', 'Unnamed: 0', C.HARMONIZED_LABEL))
lab_sc = harm[C.HARMONIZED_LABEL].reindex(H.index).astype(str)
is_cells = (light.obs.loc[H.index, 'arm'].astype(str) == 'cells').values
usage_pop = C.usage_shares(H[is_cells]).groupby(lab_sc[is_cells].values).mean()
usage_pop = usage_pop.loc[[s for s in C.FB_STATES if s in usage_pop.index]]
print('scRNA-seq cells per state:', lab_sc[is_cells].value_counts().reindex(usage_pop.index).to_dict())

xen = centre_within_donor(t1[(t1['representation'] == 'cells') & (t1['level'] == 'taa_l3_upd')])
prof_x = xen[xen['group'].isin(C.FB_STATES)].pivot_table(index='group', columns='source', values='score_centred', aggfunc='mean')
states = [s for s in C.FB_STATES if s in prof_x.index and s in usage_pop.index]
prof_x = prof_x.reindex(index=states, columns=FACTORS)
prof_s = usage_pop.reindex(index=states, columns=FACTORS)
prof_x, prof_s = prof_x.sub(prof_x.mean(axis=0), axis=1), prof_s.sub(prof_s.mean(axis=0), axis=1)
flat = spearmanr(prof_x.to_numpy().ravel(), prof_s.to_numpy().ravel())
gate1 = pd.DataFrame({'name': [NAMES[f] for f in FACTORS], 'xenium_top_state': prof_x.idxmax(axis=0), 'scrnaseq_top_state': prof_s.idxmax(axis=0),
                      'rho_across_states': [float(spearmanr(prof_x[f], prof_s[f]).statistic) for f in FACTORS]})
gate1['agree'] = gate1['xenium_top_state'] == gate1['scrnaseq_top_state']
gate1.to_csv(f'{RES}/gate1_crossmodality.csv')
print(f'states compared: {states}')
print(f'GATE 1: Spearman {flat.statistic:.3f} (p = {flat.pvalue:.2g}) over {prof_x.size} state x factor pairs; top state agrees for {int(gate1["agree"].sum())}/16')
display(gate1.round(3))

fig, axes = plt.subplots(1, 3, layout='constrained', figsize=C.figsize_mm('double', 84), gridspec_kw={'width_ratios': [1, 1, 0.8]})
for ax, P, ttl, lab in [(axes[0], prof_x, 'Xenium (panel signatures, centred ULM)', 'centred ULM t'),
                        (axes[1], prof_s, 'scRNA-seq (iNMF usage, centred)', 'centred usage')]:
    vmax = float(np.nanmax(np.abs(P.to_numpy())))
    im = C.heatmap(ax, P.T, C.CMAP_DIV, -vmax, vmax)
    ax.set_yticklabels([LAB[f] for f in P.columns]); ax.set_title(ttl)
    fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02, label=lab)
axes[2].scatter(prof_s.to_numpy().ravel(), prof_x.to_numpy().ravel(), s=6, color=C.COLORS['blue'], lw=0)
axes[2].axhline(0, color='0.7', lw=0.5); axes[2].axvline(0, color='0.7', lw=0.5); axes[2].grid(False)
axes[2].set_xlabel('scRNA-seq centred usage'); axes[2].set_ylabel('Xenium centred ULM'); axes[2].set_title(f'ρ = {flat.statistic:.2f} (n = {prof_x.size})')
C.save(fig, f'{PLT}/gate1_crossmodality.pdf')
plt.show()

scRNA-seq cells per state: {'Fb-perilo': 23063, 'Fb-interlo': 18363, 'Fb-interm': 14815, 'Fb-med': 7517, 'Fb-adipo': 5216, 'Fb-TGFB+': 1284}
states compared: ['Fb-perilo', 'Fb-interlo', 'Fb-interm', 'Fb-med', 'Fb-adipo', 'Fb-TGFB+']
GATE 1: Spearman 0.559 (p = 3.3e-09) over 96 state x factor pairs; top state agrees for 11/16


,name,xenium_top_state,scrnaseq_top_state,rho_across_states,agree
S1,TGFb / hypoxia / mechano,Fb-interlo,Fb-TGFB+,0.143,False
S2,lymphoid (contamination),Fb-interm,Fb-med,-0.314,False
S3,interlobular (CCN5),Fb-interlo,Fb-interlo,0.600,True
S4,proliferation,Fb-TGFB+,Fb-TGFB+,-0.143,True
S5,perilobular,Fb-perilo,Fb-perilo,0.600,True
S6,IEG / dissociation,Fb-interlo,Fb-med,-0.086,False
S7,inflammatory,Fb-adipo,Fb-interm,0.829,False
S8,perilobular (nuclear),Fb-perilo,Fb-perilo,0.829,True
S9,interlobular (PI16),Fb-interlo,Fb-interlo,0.771,True
S10,ECM synthesis,Fb-interlo,Fb-TGFB+,0.600,False


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/gate1_crossmodality.pdf   183 x 84 mm


## 5 · Gate 2 — which cell type carries each signature in Xenium?

`fb_margin` = best fibroblast state (or adipocyte) − best other cell type, centred ULM on the
donor × cell-type pseudobulk. A programme with a negative margin is carried by another lineage on
this panel; its all-cell niche map is still a valid tissue map, but is not fibroblast biology and
it gets no fibroblast-only reading.

In [6]:
spec = xen.pivot_table(index='source', columns='group', values='score_centred', aggfunc='mean').reindex(FACTORS)
fb_cols = [c for c in spec.columns if c in C.FB_STATES_EXT]
other = [c for c in spec.columns if c not in C.FB_STATES_EXT]
gate2 = pd.DataFrame({'name': [NAMES[f] for f in spec.index], 'best_fb_type': spec[fb_cols].idxmax(axis=1), 'best_fb': spec[fb_cols].max(axis=1),
                      'best_other_type': spec[other].idxmax(axis=1), 'best_other': spec[other].max(axis=1)})
gate2['fb_margin'] = gate2['best_fb'] - gate2['best_other']
gate2['fb_readable'] = gate2['fb_margin'] > 0
gate2['crossmodality_rho'] = gate1['rho_across_states'].reindex(gate2.index)
gate2.to_csv(f'{RES}/gate2_specificity.csv')
READABLE = list(gate2.index[gate2['fb_readable']])
display(gate2.sort_values('fb_margin', ascending=False).round(2))
print(f'GATE 2: fibroblast-readable {len(READABLE)}/16; not readable: ' + ', '.join(f'{f} -> {gate2.loc[f, "best_other_type"]}' for f in gate2.index[~gate2['fb_readable']]))

d = gate2.sort_values('fb_margin')
fig, ax = plt.subplots(layout='constrained', figsize=C.figsize_mm('one-half', 4.6 * len(d) + 18))
ax.barh(np.arange(len(d)), d['fb_margin'], height=0.7, color=np.where(d['fb_readable'], C.COLORS['blue'], C.COLORS['magenta']))
ax.set_yticks(np.arange(len(d))); ax.set_yticklabels([LAB[f] for f in d.index]); ax.axvline(0, color='0.2', lw=0.6); ax.grid(False)
for y, f in enumerate(d.index):
    ax.text(max(d.loc[f, 'fb_margin'], 0) + 0.15, y, f'{d.loc[f, "best_fb_type"]} vs {d.loc[f, "best_other_type"]}', va='center', ha='left', color='0.3')
ax.set_xlim(d['fb_margin'].min() - 0.5, d['fb_margin'].max() + 9)
ax.set_xlabel('best fibroblast state − best other cell type (centred ULM t)'); ax.set_title('Is the signature fibroblast-readable in Xenium?')
C.save(fig, f'{PLT}/gate2_specificity.pdf')
plt.show()

,name,best_fb_type,best_fb,best_other_type,best_other,fb_margin,fb_readable,crossmodality_rho
source,,,,,,,,
S15,mature adipose,Adipocyte,16.48,EC-cap_lipid,6.73,9.75,True,0.83
S9,interlobular (PI16),Fb-interlo,7.99,SMC,1.96,6.03,True,0.77
S3,interlobular (CCN5),Fb-interlo,8.96,EC-lymphatic,3.09,5.87,True,0.60
S11,hormone / vascular tone,Fb-adipo,9.32,Pericyte,4.37,4.95,True,0.83
S10,ECM synthesis,Fb-interlo,8.14,Pericyte,3.69,4.45,True,0.60
S5,perilobular,Fb-perilo,7.16,SMC,3.35,3.81,True,0.60
S8,perilobular (nuclear),Fb-perilo,4.99,Pericyte,3.09,1.90,True,0.83
S7,inflammatory,Fb-adipo,4.61,EC-peri,3.30,1.31,True,0.83
S12,inflammatory stress,Fb-interlo,3.11,Mac,2.57,0.54,True,0.71


GATE 2: fibroblast-readable 9/16; not readable: S1 -> EC-lymphatic, S2 -> T_αβT(entry), S4 -> Myeloid_dev, S6 -> EC-ven, S13 -> SMC, S14 -> mTECII, S16 -> EC-peri


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/gate2_specificity.pdf   130 x 92 mm


## 6 · Niche statistics: donor agreement on the niche ordering (Kendall's W) and age contrasts

In [7]:
def kendall_w(M):
    m, k = M.shape
    R = np.apply_along_axis(rankdata, 1, M)
    Rj = R.sum(axis=0)
    return 12 * float(((Rj - Rj.mean()) ** 2).sum()) / (m ** 2 * (k ** 3 - k)), R.mean(axis=0)


_NULL = {}


def perm_null(m, k, n_perm, seed=0):
    """Friedman chi2 null for an m x k block design by permuting ranks within each donor (design-only, cached)."""
    key = (m, k, n_perm, seed)
    if key not in _NULL:
        rng = np.random.default_rng(seed)
        R = rng.permuted(np.tile(np.arange(1, k + 1, dtype=float), (n_perm, m, 1)), axis=2)
        Rj = R.sum(axis=1)
        Wn = 12 * ((Rj - Rj.mean(axis=1, keepdims=True)) ** 2).sum(axis=1) / (m ** 2 * (k ** 3 - k))
        _NULL[key] = m * (k - 1) * Wn
    return _NULL[key]


def pstars(p):
    return '' if not np.isfinite(p) else ('***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')


def niche_stats_within_age(t, sources, level, n_perm=10_000, seed=0, alpha=0.05):
    """Do the niches differ within one age group? Donors are blocks, niches treatments (complete blocks only)."""
    order = C.NICHE_ORDERS[level]
    srows, erows = [], []
    for age in C.AGE_ORDER:
        sub = t[(t['age_group'].astype(str) == age) & (t['level'] == level)]
        if sub.empty:
            continue
        donors = sorted(sub['donor'].unique())
        counts = sub.groupby('group', observed=True)['donor'].nunique()
        niches = [n for n in order if counts.get(n, 0) == len(donors)]
        if len(niches) < 3:
            continue
        for s in sources:
            P = sub[sub['source'] == s].pivot_table(index='donor', columns='group', values='score').reindex(index=donors, columns=niches)
            if P.isna().any().any():
                continue
            M = P.to_numpy(float)
            m, k = M.shape
            Wk, mean_rank = kendall_w(M)
            chi2 = m * (k - 1) * Wk
            p_perm = (1 + int((perm_null(m, k, n_perm, seed) >= chi2 - 1e-12).sum())) / (n_perm + 1)
            centred = M - M.mean(axis=1, keepdims=True)
            mc = centred.mean(axis=0)
            srows.append({'age_group': age, 'source': s, 'n_donors': m, 'n_niches': k, 'kendall_w': Wk, 'friedman_chi2': chi2, 'pvalue': p_perm,
                          'top_niche': niches[int(np.argmax(mc))], 'bottom_niche': niches[int(np.argmin(mc))], 'range_centred': float(mc.max() - mc.min())})
            for j, nb in enumerate(niches):
                col = centred[:, j]
                erows.append({'age_group': age, 'source': s, 'group': nb, 'mean_centred': float(col.mean()), 'n_donors': m,
                              'n_donors_positive': int((col > 0).sum()), 'consistent': bool(np.all(col > 0) or np.all(col < 0))})
    stats = pd.DataFrame(srows)
    if len(stats):
        stats['padj'] = stats.groupby('age_group')['pvalue'].transform(lambda p: multipletests(p, method='fdr_bh')[1])
        stats['significant'] = stats['padj'] < alpha
        stats['stars'] = stats['padj'].map(pstars)
    return stats, pd.DataFrame(erows)


def young_vs_adult(t, gkeys, value='score'):
    """Adult (3) vs young (4) per group: delta of means, rank-biserial effect_r, asymptotic KW p (BH within source), exact MWU p."""
    rows = []
    for keys, d in t.groupby(gkeys, observed=True):
        y = d.loc[d['age_group2'] == 'young', value].to_numpy(float)
        a = d.loc[d['age_group2'] == 'adult', value].to_numpy(float)
        if len(y) < 2 or len(a) < 2:
            continue
        try:
            p_kw = float(kruskal(y, a)[1])
        except ValueError:
            p_kw = np.nan
        U_ = float(mannwhitneyu(a, y, alternative='two-sided')[0])
        try:
            p_exact = float(mannwhitneyu(a, y, alternative='two-sided', method='exact')[1])
        except Exception:
            p_exact = np.nan
        rows.append({**dict(zip(gkeys, keys)), 'n_young': len(y), 'n_adult': len(a), 'mean_young': y.mean(), 'mean_adult': a.mean(),
                     'delta_adult_vs_young': a.mean() - y.mean(), 'effect_r': 2 * U_ / (len(a) * len(y)) - 1, 'p_kw': p_kw, 'p_exact': p_exact})
    t2 = pd.DataFrame(rows)
    t2['padj_kw'] = t2.groupby(['representation', 'level', 'source'], observed=True)['p_kw'].transform(lambda p: multipletests(p.fillna(1), method='fdr_bh')[1])
    return t2


def niche_matrix(df, factors, value='score'):
    """Factors x niches: mean over donors of the within-donor centred score, and the donor count per cell."""
    d = df[df['source'].isin(factors)].copy()
    d['_v'] = d.groupby(['source', 'donor'], observed=True)[value].transform(lambda s_: s_ - s_.mean())
    M = d.pivot_table(index='source', columns='group', values='_v', aggfunc='mean', observed=True)
    n = d.pivot_table(index='source', columns='group', values='_v', aggfunc='count', observed=True)
    cols = [c for c in C.NICHE_ORDER if c in M.columns]
    return M.reindex(index=list(factors), columns=cols), n.reindex(index=list(factors), columns=cols)


stats_all, eff_all = [], []
for (rep, lv) in NICHE_UNITS:
    st, ef = niche_stats_within_age(t1[t1['representation'] == rep], FACTORS, lv)
    for df_ in (st, ef):
        df_['representation'], df_['level'] = rep, lv
    stats_all.append(st); eff_all.append(ef)
stats = pd.concat(stats_all, ignore_index=True)
effects = pd.concat(eff_all, ignore_index=True)
stats.to_csv(f'{RES}/table2_niche_stats.csv', index=False)
effects.to_csv(f'{RES}/table3_niche_effects.csv', index=False)
s2 = stats[stats['level'] == 'morpho_niche_l2']
print('factors with a donor-consistent niche ordering (Kendall W, BH padj < 0.05 within age group), morpho_niche_l2:')
print(s2.groupby(['representation', 'age_group'], observed=True)['significant'].sum().unstack('age_group').to_string())


def stars_by_age(rep, level='morpho_niche_l2'):
    sub = stats[(stats['representation'] == rep) & (stats['level'] == level)]
    return {C.AGE_TITLE[a]: dict(zip(sub.loc[sub['age_group'] == a, 'source'], sub.loc[sub['age_group'] == a, 'stars'])) for a in C.AGE_ORDER}

factors with a donor-consistent niche ordering (Kendall W, BH padj < 0.05 within age group), morpho_niche_l2:
age_group       adult  epaed  lpaed
representation                     
cells              16      3     12
fbcells            14      0      0
patches            16      4     12


## 7 · All sixteen programmes across the morphological compartments

For each representation: one panel per age group (dot area = donors agreeing with the sign,
stars = Kendall-W padj for the niche ordering), then all seven sections pooled. Scores are
centred within donor across the eight niches.

In [8]:
def dotplot_by_age(rep, factors, tag, level='morpho_niche_l2', cell_mm=5.2):
    """Three age-group panels side by side (double width), then all sections pooled (one-half width)."""
    sub = t1[(t1['representation'] == rep) & (t1['level'] == level)]
    frames = {C.AGE_TITLE[a]: sub[sub['age_group'].astype(str) == a] for a in C.AGE_ORDER if (sub['age_group'].astype(str) == a).any()}
    fig, tabs = C.niche_dotplot(frames, factors, C.NICHE_ORDERS[level], labels=LAB, row_annot=stars_by_age(rep, level),
                                width='double', cell_mm=cell_mm, suptitle=f'{tag} — {C.REP_NAME[rep]}')
    C.save(fig, f'{PLT}/niche_dotplot_{tag.replace(" ", "")}_{rep}_{level}_byAge.pdf')
    plt.show()
    out = pd.concat([tabs[a][0].assign(age_group=a) for a in tabs])
    out.to_csv(f'{RES}/niche_matrix_{tag.replace(" ", "")}_{rep}_{level}_byAge.csv')
    fig, tabs = C.niche_dotplot({'all sections': sub}, factors, C.NICHE_ORDERS[level], labels=LAB, width='one-half', cell_mm=cell_mm,
                                suptitle=f'{tag} — {C.REP_NAME[rep]}, all 7 sections')
    C.save(fig, f'{PLT}/niche_dotplot_{tag.replace(" ", "")}_{rep}_{level}_pooled.pdf')
    plt.show()
    tabs['all sections'][0].to_csv(f'{RES}/niche_matrix_{tag.replace(" ", "")}_{rep}_{level}_pooled.csv')


for rep in ['patches', 'cells', 'fbcells']:
    dotplot_by_age(rep, FACTORS, 'S1-S16')

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1-S16_patches_morpho_niche_l2_byAge.pdf   183 x 93 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1-S16_patches_morpho_niche_l2_pooled.pdf   130 x 113 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1-S16_cells_morpho_niche_l2_byAge.pdf   183 x 93 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1-S16_cells_morpho_niche_l2_pooled.pdf   130 x 113 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1-S16_fbcells_morpho_niche_l2_byAge.pdf   183 x 93 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1-S16_fbcells_morpho_niche_l2_pooled.pdf   130 x 113 mm


## 8 · The four selected programmes: S1, S7, S12, S15

Dotplots as above, and plain heatmaps of the adult sections (A67, A72, A70) in two versions: one
shared colour scale (magnitudes comparable between programmes) and each programme divided by its
own |max| (the niche *pattern*, magnitude discarded — S15 spans several times the range of S7 and
washes it out on a shared scale). **S1 is not fibroblast-readable** (gate 2: its best Xenium cell
type is a non-fibroblast lineage), so its fibroblast-only row is a weak reading and is marked.

In [9]:
for rep in ['patches', 'cells', 'fbcells']:
    dotplot_by_age(rep, SELECTED, 'S1 S7 S12 S15', cell_mm=6.5)

SEL_LAB = {f: LAB[f] + ('' if gate2.loc[f, 'fb_readable'] else ' †') for f in SELECTED}
for rep in ['patches', 'cells', 'fbcells']:
    sub = t1[(t1['representation'] == rep) & (t1['level'] == 'morpho_niche_l2') & (t1['age_group'].astype(str) == 'adult')]
    M, n = niche_matrix(sub, SELECTED)
    n_max = np.nanmax(n.values)
    annot = n.map(lambda v: '' if not np.isfinite(v) or v >= n_max else str(int(v)))
    M.to_csv(f'{RES}/niche_matrix_selected_{rep}_adult.csv')
    for scaled, sfx in [(False, ''), (True, '_rowScaled')]:
        Mp = M.div(M.abs().max(axis=1).replace(0, np.nan), axis=0) if scaled else M
        fig = C.niche_heatmap(Mp, labels=SEL_LAB, vlim=1.0 if scaled else None, annot=annot,
                              cbar_label='score / |max| per programme' if scaled else 'mean score\n(centred within donor)',
                              suptitle=f'S1 / S7 / S12 / S15 — adult sections, {C.REP_NAME[rep]}' + (' († not fibroblast-readable)' if rep == 'fbcells' else ''))
        C.save(fig, f'{PLT}/niche_heatmap_selected_{rep}_adult{sfx}.pdf')
        plt.show()
    print(rep); print(M.round(2).to_string(), '\n')

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1S7S12S15_patches_morpho_niche_l2_byAge.pdf   183 x 46 mm
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1S7S12S15_patches_morpho_niche_l2_pooled.pdf   130 x 56 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1S7S12S15_cells_morpho_niche_l2_byAge.pdf   183 x 46 mm
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1S7S12S15_cells_morpho_niche_l2_pooled.pdf   130 x 56 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1S7S12S15_fbcells_morpho_niche_l2_byAge.pdf   183 x 46 mm
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_dotplot_S1S7S12S15_fbcells_morpho_niche_l2_pooled.pdf   130 x 56 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_heatmap_selected_patches_adult.pdf   89 x 51 mm
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_heatmap_selected_patches_adult_rowScaled.pdf   89 x 51 mm
patches
group   cortex_septal  cortex  medulla  medulla_hc  adipose  fibrosis_adipose  fibrosis  vessel
source                                                                                         
S1              -0.42   -0.44    -0.58       -0.28     0.24              0.54      0.45    0.48
S7              -0.80   -0.93    -1.45       -1.77     0.70              1.74      1.47    1.03
S12             -0.66   -0.45    -0.93       -0.72     0.07              1.26      0.96    0.46
S15             -1.48   -0.09    -1.74       -2.46     3.20              2.23      0.76   -0.42 



-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_heatmap_selected_cells_adult.pdf   89 x 51 mm
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_heatmap_selected_cells_adult_rowScaled.pdf   89 x 51 mm
cells
group   cortex_septal  cortex  medulla  medulla_hc  adipose  fibrosis_adipose  fibrosis  vessel
source                                                                                         
S1              -0.41   -0.45    -0.57       -0.26     0.26              0.45      0.48    0.50
S7              -0.77   -0.96    -1.44       -1.74     0.70              1.73      1.45    1.05
S12             -0.63   -0.47    -0.93       -0.72     0.10              1.21      0.97    0.47
S15             -1.18   -0.12    -1.50       -2.10     2.88              1.69      0.54   -0.21 



-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_heatmap_selected_fbcells_adult.pdf   89 x 51 mm
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/niche_heatmap_selected_fbcells_adult_rowScaled.pdf   89 x 51 mm
fbcells
group   cortex_septal  cortex  medulla  medulla_hc  adipose  fibrosis_adipose  fibrosis  vessel
source                                                                                         
S1              -0.78    0.33    -0.42       -0.06    -0.20              0.08      0.74    0.31
S7              -0.47    0.38    -0.86       -1.30     0.04              0.66      0.94    0.61
S12             -0.71    0.13    -0.86       -1.09    -0.35              1.15      1.27    0.45
S15             -2.19    2.52    -1.19       -4.59     4.53              1.35      0.59   -1.03 



## 9 · Age: adult − young per niche, all cells versus fibroblasts only

Where a programme's age change survives restriction to the fibroblast compartment, the programme
changes *within* fibroblasts; where it collapses or flips sign, the all-cell change was
fibroblasts arriving, not fibroblasts changing. Stars: Kruskal–Wallis padj (indicative only with
3 vs 4 sections); dot: complete separation of the age groups (|effect_r| = 1).

In [10]:
t2 = young_vs_adult(t1, ['representation', 'level', 'group', 'source'])
t2.to_csv(f'{RES}/table4_age_stats.csv', index=False)


def plot_delta(rep, level, sources, title, save):
    d2 = t2[(t2['representation'] == rep) & (t2['level'] == level) & t2['source'].isin(sources)]
    groups = [g for g in C.NICHE_ORDERS[level] if g in set(d2['group'])]
    delta = d2.pivot_table(index='source', columns='group', values='delta_adult_vs_young').reindex(index=list(sources), columns=groups)
    padj = d2.pivot_table(index='source', columns='group', values='padj_kw').reindex(index=list(sources), columns=groups)
    er = d2.pivot_table(index='source', columns='group', values='effect_r').reindex(index=list(sources), columns=groups)
    ann = padj.map(pstars) + pd.DataFrame(np.where(er.abs().to_numpy() == 1, '•', ''), index=delta.index, columns=delta.columns)
    vmax = float(np.nanmax(np.abs(delta.to_numpy())))
    fig, ax = plt.subplots(layout='constrained', figsize=C.figsize_mm('single', 5.0 * len(sources) + 30))
    im = C.heatmap(ax, delta, C.CMAP_DIV, -vmax, vmax, annot=ann, annot_fontsize=5)
    ax.set_yticklabels([LAB[s] for s in sources]); ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02, label='Δ (adult − young)')
    C.save(fig, save)
    plt.show()


for rep in ['patches', 'cells', 'fbcells']:
    plot_delta(rep, 'morpho_niche_l2', FACTORS, f'adult − young — {C.REP_NAME[rep]}', f'{PLT}/age_delta_{rep}_morpho_niche_l2.pdf')

A = t2[(t2['representation'] == 'cells') & (t2['level'] == 'morpho_niche_l2')].set_index(['source', 'group'])[['delta_adult_vs_young']]
B = t2[(t2['representation'] == 'fbcells') & (t2['level'] == 'morpho_niche_l2')].set_index(['source', 'group'])[['delta_adult_vs_young']]
J = A.join(B, lsuffix='_allcells', rsuffix='_fbonly', how='inner').dropna()
J['sign_flip'] = np.sign(J['delta_adult_vs_young_allcells']) != np.sign(J['delta_adult_vs_young_fbonly'])
summ = J.reset_index().groupby('source').agg(delta_allcells=('delta_adult_vs_young_allcells', 'mean'), delta_fbonly=('delta_adult_vs_young_fbonly', 'mean'),
                                              n_flips=('sign_flip', 'sum')).reindex(FACTORS)
summ['name'] = [NAMES[f] for f in summ.index]
summ['fb_readable'] = gate2['fb_readable'].reindex(summ.index)
summ.to_csv(f'{RES}/age_delta_allcells_vs_fbonly.csv')
rho = spearmanr(J['delta_adult_vs_young_allcells'], J['delta_adult_vs_young_fbonly'])
print(f'{len(J)} matched factor x niche contrasts: Spearman(Δ all cells, Δ fibroblasts only) = {rho.statistic:.2f}; sign flips {int(J["sign_flip"].sum())}')
print('\nmean adult − young over the eight niches:')
print(summ.round(2).to_string())

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/age_delta_patches_morpho_niche_l2.pdf   89 x 110 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/age_delta_cells_morpho_niche_l2.pdf   89 x 110 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/age_delta_fbcells_morpho_niche_l2.pdf   89 x 110 mm
96 matched factor x niche contrasts: Spearman(Δ all cells, Δ fibroblasts only) = 0.65; sign flips 32

mean adult − young over the eight niches:
        delta_allcells  delta_fbonly  n_flips                         name  fb_readable
source                                                                                 
S1                0.83          0.76        0     TGFb / hypoxia / mechano        False
S2               -4.58         -3.26        0     lymphoid (contamination)        False
S3                1.72          1.29        1          interlobular (CCN5)         True
S4               -7.54         -0.97        0                proliferation        False
S5                1.48         -3.28        6                  perilobular         True
S6                1.89          0.86        0           IEG / dissociation        Fal

## 10 · Composition control

Spearman between each programme's niche score and the fibroblast + adipocyte fraction of that
donor × niche. A high |ρ| in the all-cell map means the map is largely fibroblast abundance; the
same number on the fibroblast-only map says how much of that channel is left (it cannot reach
zero, because niches still differ in *which* fibroblast state they hold).

In [11]:
comp = pd.read_csv(sorted(glob.glob(C.XENIUM_COMPOSITION_GLOB))[-1])
fa = (comp[(comp['niche_level'] == 'morpho_niche_l2') & (comp['ctype_level'] == 'taa_l2_upd') & comp['ctype'].isin(['Fb', 'Adipocyte'])]
      .groupby(['donor', 'niche'])['frac'].sum().rename('fb_adipo_fraction').reset_index())
rows = []
for rep in ['patches', 'cells', 'fbcells']:
    base = t1[(t1['representation'] == rep) & (t1['level'] == 'morpho_niche_l2')].merge(fa, left_on=['donor', 'group'], right_on=['donor', 'niche'], how='left')
    for f, d in base.groupby('source', observed=True):
        dd = d[['score', 'fb_adipo_fraction']].dropna()
        rows.append({'representation': rep, 'source': f, 'rho_fb_fraction': float(spearmanr(dd['score'], dd['fb_adipo_fraction']).statistic)})
cc = pd.DataFrame(rows).pivot_table(index='source', columns='representation', values='rho_fb_fraction').reindex(FACTORS)
cc.to_csv(f'{RES}/composition_control.csv')
print(cc.round(2).to_string())

fig, ax = plt.subplots(layout='constrained', figsize=C.figsize_mm('one-half', 4.6 * 16 + 18))
y = np.arange(16)
ax.barh(y - 0.2, cc['cells'], height=0.38, color=C.COLORS['magenta'], label='all cells')
ax.barh(y + 0.2, cc['fbcells'], height=0.38, color=C.COLORS['blue'], label='fibroblasts only')
ax.set_yticks(y); ax.set_yticklabels([LAB[f] for f in cc.index]); ax.invert_yaxis(); ax.axvline(0, color='0.3', lw=0.6); ax.set_xlim(-1, 1); ax.grid(False)
ax.set_xlabel('Spearman(niche score, fibroblast + adipocyte fraction)'); ax.legend(frameon=False, loc='lower right')
C.save(fig, f'{PLT}/composition_control.pdf')
plt.show()

representation  cells  fbcells  patches
source                                 
S1               0.64    -0.16     0.64
S2              -0.67    -0.83    -0.68
S3               0.87     0.49     0.89
S4              -0.63    -0.70    -0.63
S5               0.90    -0.51     0.95
S6               0.71     0.31     0.73
S7               0.73     0.57     0.73
S8               0.67    -0.63     0.68
S9               0.91     0.20     0.91
S10              0.60    -0.44     0.64
S11              0.90     0.65     0.90
S12              0.70     0.41     0.71
S13              0.65    -0.81     0.62
S14              0.12    -0.77     0.10
S15              0.66     0.69     0.67
S16              0.76     0.33     0.77


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/05_xenium_spatial/composition_control.pdf   130 x 92 mm


## 11 · Summary

In [12]:
summary = pd.DataFrame({'name': pd.Series(NAMES), 'n_genes': sigtab['n_genes'], 'top50_on_panel': sigtab['top50_on_panel'],
                        'gate1_rho': gate1['rho_across_states'], 'gate1_top_state_agrees': gate1['agree'],
                        'gate2_fb_margin': gate2['fb_margin'], 'fb_readable': gate2['fb_readable'],
                        'rho_fb_fraction_allcells': cc['cells'], 'rho_fb_fraction_fbonly': cc['fbcells'],
                        'delta_allcells_l2': summ['delta_allcells'], 'delta_fbonly_l2': summ['delta_fbonly']}).reindex(FACTORS)
summary.to_csv(f'{RES}/summary.csv')
print(summary.round(2).to_string())
print(f'\nRead niche and age claims from the FIBROBLAST-ONLY map for the {len(READABLE)} fibroblast-readable programmes; '
      'the all-cell and patch maps are the tissue-level view and are largely fibroblast abundance where |rho_fb_fraction_allcells| is high.')

                            name  n_genes  top50_on_panel  gate1_rho  gate1_top_state_agrees  gate2_fb_margin  fb_readable  rho_fb_fraction_allcells  rho_fb_fraction_fbonly  delta_allcells_l2  delta_fbonly_l2
S1      TGFb / hypoxia / mechano       30              23       0.14                   False            -1.00        False                      0.64                   -0.16               0.83             0.76
S2      lymphoid (contamination)       30              25      -0.31                   False            -9.35        False                     -0.67                   -0.83              -4.58            -3.26
S3           interlobular (CCN5)       30              21       0.60                    True             5.87         True                      0.87                    0.49               1.72             1.29
S4                 proliferation       30              31      -0.14                    True            -4.84        False                     -0.63                

## Files written

| file | contents |
|---|---|
| `results/final/05_xenium_spatial/signatures.csv` | the 30-gene panel signatures and coverage per factor |
| `results/final/05_xenium_spatial/table1_scores.csv` | every pseudobulk × factor (ULM and score_genes) |
| `results/final/05_xenium_spatial/gate1_crossmodality.csv`, `gate2_specificity.csv` | the two gates |
| `results/final/05_xenium_spatial/table2_niche_stats.csv`, `table3_niche_effects.csv` | Kendall W per factor and age group; centred score per niche |
| `results/final/05_xenium_spatial/niche_matrix_*.csv` | the factor × niche matrices behind every dotplot |
| `results/final/05_xenium_spatial/niche_matrix_selected_*_adult.csv` | S1/S7/S12/S15 × niche, adult sections |
| `results/final/05_xenium_spatial/table4_age_stats.csv`, `age_delta_allcells_vs_fbonly.csv`, `composition_control.csv`, `summary.csv` | age and controls |
| `plots/final/05_xenium_spatial/niche_dotplot_S1-S16_*` | all 16 programmes × 8 niches, per age group and pooled, three representations |
| `plots/final/05_xenium_spatial/niche_dotplot_S1S7S12S15_*`, `niche_heatmap_selected_*` | the four selected programmes |
| `plots/final/05_xenium_spatial/gate*.pdf`, `age_delta_*.pdf`, `composition_control.pdf` | gates and controls |